# 064 — Does strategy-tag usage differ between good and bad distractor traces?

Produces Appendix C.6: the omnibus table Table 29, the gated post-hoc tables Table 30 / Table 31, and the
stage-level figures Figure 9 / Figure 10. Feeds the "Diagnosing failure modes" paragraph in Section 4.3 and
the ERR_DESC/CORR share effects it reports. See `064-methodology-explainer.md` and Appendix C.6 for the
full derivation (sampling design, PERMANOVA/Freedman-Lane, BH-FDR); this notebook is the implementation.

In [ ]:
import os, ast
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RNG = np.random.default_rng(42)
N_PERM = 10000

TAGS = ["inter","corr","link","err_desc","err_sim","inst","plaus","discr","curate","recon"]
TAG_DISPLAY_NAMES = {
    "inter":"Task Interpretation", "corr":"Correct Answer Ref.", "link":"Conceptual Link",
    "err_desc":"Error Description", "err_sim":"Error Simulation", "inst":"Outcome Instantiation",
    "plaus":"Plausibility Check", "discr":"Discriminability Check", "curate":"Final Set Curation",
    "recon":"Reconsideration",
}
MODELS   = ["deepseek","glm"]
REGIMES  = ["reasoning","cot"]
DATASETS = ["eedi","sciq"]
STEMS = {
    ("deepseek","reasoning"): "deepseek-naive-deepseek-reasoner",
    ("deepseek","cot"):       "deepseek-naive-cot-deepseek-chat",
    ("glm","reasoning"):      "openrouter-naive-z-ai_glm-4.7-reasoner",
    ("glm","cot"):            "openrouter-naive-cot-z-ai_glm-4.7-chat",
}
DATAFOLDER  = {"eedi":"eedi_data", "sciq":"sciq_data"}
REGIME_NAME = {"reasoning":"reasoning", "cot":"CoT"}
os.makedirs("figures", exist_ok=True)

In [ ]:
# ---------- loaders (no IPW: the good/bad label is the design) ----------
def parse_labels(seq_str):
    if not isinstance(seq_str, str) or not seq_str.startswith("["):
        return []
    return [lab for _, lab in ast.literal_eval(seq_str)]

def load_run(dataset, model, regime):
    """One run's annotated traces from the high/low_match_solvable strata. group=1 -> pm>0.5."""
    data_folder = DATAFOLDER[dataset]; run = STEMS[(model, regime)]
    frames = []
    for bucket, g in (("high_match_solvable", 1), ("low_match_solvable", 0)):
        path = f"{data_folder}/joint_results/annotated/{run}_{bucket}_annot_parsed.csv"
        assert "long_problems" not in path
        d = pd.read_csv(path)[["Id","proportional_match","annotation_sequence","nr_steps_of_solution"]].copy()
        d["group"], d["model"] = g, model
        frames.append(d)
    o = pd.concat(frames, ignore_index=True).drop_duplicates("Id").reset_index(drop=True)
    o["labels"] = o["annotation_sequence"].map(parse_labels)
    o["ntot"]   = o["labels"].map(len)
    o["steps"]  = pd.to_numeric(o["nr_steps_of_solution"], errors="coerce")
    return o

def load_pooled(dataset, regime):
    """Both models stacked for one (dataset, regime); keeps a model column for blocking."""
    df = pd.concat([load_run(dataset, m, regime) for m in MODELS], ignore_index=True)
    return df.reset_index(drop=True)

In [ ]:
# ---------- feature encodings ----------
def count_features(df):
    return pd.DataFrame({t: df["labels"].map(lambda L, t=t: Counter(L)[t]) for t in TAGS}, index=df.index)

def share_features(df):
    """Percentage of the trace's reasoning spent on each strategy (100 * count / total tags)."""
    cnt = count_features(df)
    ntot = df["ntot"].replace(0, np.nan).values[:, None]
    return pd.DataFrame(np.nan_to_num(100.0 * cnt.values / ntot), columns=TAGS, index=df.index)

def presence_features(df):
    return pd.DataFrame({t: df["labels"].map(lambda L, t=t: int(t in set(L))) for t in TAGS}, index=df.index)

FEATURE_BUILDERS = {"count": count_features, "share": share_features}

In [ ]:
# ---------- pooled per-tag contrast: model-adjusted good-bad Delta + Freedman-Lane permutation p ----------
def _model_dummies(model):
    """Intercept + one dummy per extra model level (drop first)."""
    levels = list(dict.fromkeys(model))
    cols = [np.ones(len(model))]
    for lev in levels[1:]:
        cols.append((model == lev).astype(float))
    return np.column_stack(cols)

def freedman_lane(f, group, Z, blocks, n_perm=N_PERM, rng=RNG):
    """Delta = OLS coefficient on `group` in f ~ Z + group; p via Freedman-Lane (Appendix C.6)."""
    f = np.asarray(f, float); group = np.asarray(group, float)
    X = np.column_stack([Z, group])                      # group is the last column
    if np.linalg.matrix_rank(X) < X.shape[1] or f.std() == 0:
        return np.nan, np.nan
    H  = np.linalg.solve(X.T @ X, X.T)                   # last row extracts the group coef
    h  = H[-1]
    b_obs = h @ f
    fhat = Z @ (np.linalg.solve(Z.T @ Z, Z.T) @ f)       # reduced model f ~ Z
    e = f - fhat
    c = h @ fhat
    block_idx = [np.where(blocks == b)[0] for b in np.unique(blocks)]
    null = np.empty(n_perm)
    for k in range(n_perm):
        ep = e.copy()
        for idx in block_idx:
            ep[idx] = rng.permutation(ep[idx])
        null[k] = c + h @ ep
    p = (1 + np.sum(np.abs(null) >= abs(b_obs) - 1e-12)) / (1 + n_perm)
    return b_obs, p

def pooled_contrast(f, df, adjust_steps=False):
    """(delta, p) for a feature vector f over a pooled (dataset,regime) df, model-blocked."""
    group = df["group"].values.astype(float)
    model = df["model"].values
    Z = _model_dummies(model)
    if adjust_steps:
        s = df["steps"].values.astype(float)
        if np.isfinite(s).sum() <= 5 or np.nanstd(s[np.isfinite(s)]) == 0:
            return np.nan, np.nan
        m = np.isfinite(s)
        return freedman_lane(f[m], group[m], np.column_stack([Z[m], s[m]]), model[m])
    return freedman_lane(f, group, Z, model)

def bh_fdr(pvals):
    p = np.asarray(pvals, float)
    idx = [i for i in np.argsort(np.where(np.isnan(p), np.inf, p)) if not np.isnan(p[i])]
    m = len(idx); q = np.full_like(p, np.nan); prev = 1.0
    for rank, i in enumerate(reversed(idx), start=1):
        k = m - rank + 1
        prev = min(prev, p[i] * m / k); q[i] = prev
    return q

In [ ]:
# ---------- Stage 1 omnibus: PERMANOVA (pseudo-F on Bray-Curtis dissimilarity) ----------
def bray_curtis_D2(X):
    """Squared Bray-Curtis dissimilarity matrix for rows of X (trace-by-strategy count profiles)."""
    S = X.sum(1); n = len(X); D = np.zeros((n, n))
    for i in range(n):
        den = S[i] + S; den[den == 0] = 1.0
        D[i] = np.abs(X[i] - X).sum(1) / den
    return D ** 2

def permanova(D2, group, blocks, n_perm=N_PERM, rng=RNG):
    """Anderson pseudo-F for the group factor; restricted permutation within `blocks`."""
    N = len(group)
    def F(lv):
        n1 = lv.sum(); n0 = N - n1
        if n1 == 0 or n0 == 0: return 0.0
        SST = D2.sum() / (2 * N)
        c = 1.0 - lv
        SSW = (lv @ D2 @ lv) / (2 * n1) + (c @ D2 @ c) / (2 * n0)
        return (SST - SSW) / (SSW / (N - 2)) if SSW > 0 else 0.0
    l = group.astype(float); F_obs = F(l)
    block_idx = [np.where(blocks == b)[0] for b in np.unique(blocks)]
    cnt = 0
    for _ in range(n_perm):
        lp = l.copy()
        for idx in block_idx:
            lp[idx] = rng.permutation(lp[idx])
        if F(lp) >= F_obs - 1e-12: cnt += 1
    return F_obs, (1 + cnt) / (1 + n_perm)

def step_bins(df, q=3):
    """Tercile bins of nr_steps_of_solution (difficulty-robust restricted permutation)."""
    s = df["steps"].values
    try:
        return pd.qcut(pd.Series(s), q=q, duplicates="drop").astype(str).fillna("na").values
    except Exception:
        return np.where(np.isfinite(s), s.astype("U16"), "na")

In [ ]:
# ---------- run the omnibus for every (dataset, regime) ----------
omni = []
for dataset in DATASETS:
    for regime in REGIMES:
        df = load_pooled(dataset, regime)
        X = count_features(df).values.astype(float)
        D2 = bray_curtis_D2(X)
        F, p = permanova(D2, df["group"].values, df["model"].values)
        rec = dict(dataset=dataset, regime=regime, n=len(df), pseudoF=F, p=p, p_diffrobust=np.nan)
        if dataset == "eedi":  # difficulty-robust: permute within model x step-tercile blocks
            blk = np.char.add(df["model"].values.astype("U16"), step_bins(df))
            _, pdr = permanova(D2, df["group"].values, blk)
            rec["p_diffrobust"] = pdr
        omni.append(rec)
OMNI = pd.DataFrame(omni)
OMNI["q"] = bh_fdr(OMNI["p"].values)
OMNI["sig"] = OMNI["p"] < 0.05
with pd.option_context("display.float_format", lambda x: f"{x:.4f}"):
    print("STAGE 1 — OMNIBUS PERMANOVA (good vs bad strategy profile)\n")
    print(OMNI.to_string(index=False))
print("\nGated in (omnibus p<0.05):", [f"{d}/{r}" for d,r,s in zip(OMNI.dataset,OMNI.regime,OMNI.sig) if s])

In [ ]:
# ---------- sanity checks ----------
for dataset in DATASETS:
    for regime in REGIMES:
        for model in MODELS:
            df = load_run(dataset, model, regime); vc = dict(df["group"].value_counts())
            print(f"[{dataset:4s}/{regime:9s}/{model:8s}] n={len(df):3d} high={vc.get(1,0)} low={vc.get(0,0)}"
                  f" steps_ok={int(np.isfinite(df['steps']).sum())}")

print("\nPresence-contrast significant cells (naive p<0.05), all runs:")
nsig = 0
for dataset in DATASETS:
    for regime in REGIMES:
        df = load_pooled(dataset, regime); P = presence_features(df)
        for t in TAGS:
            _, p = pooled_contrast(P[t].values, df)
            if np.isfinite(p) and p < 0.05: nsig += 1
print("  ", nsig, "(presence is saturated -> we use count & share, not presence)")

_df = load_pooled("eedi","reasoning")
_d,_p = pooled_contrast(RNG.normal(size=len(_df)), _df)
print(f"\nnull check: random feature Delta={_d:+.3f}, p={_p:.2f} (expect p not small)")

## Stage 2 — post-hoc: which strategies separate good from bad (gated to significant settings)

Per-strategy model-adjusted good−bad Δ (count/share × naive/step-adjusted), BH-FDR across the ten
strategies. See Appendix C.6.

In [ ]:
# ---------- run gated per-tag contrasts ----------
GATED = [(d, r) for d, r, s in zip(OMNI.dataset, OMNI.regime, OMNI.sig) if s]

def run_stage2():
    recs = []
    for (dataset, regime) in GATED:
        df = load_pooled(dataset, regime)
        permodel = {m: load_run(dataset, m, regime) for m in MODELS}
        for kind, build in FEATURE_BUILDERS.items():
            F = build(df); Fm = {m: build(permodel[m]) for m in MODELS}
            rows = []
            for t in TAGS:
                dn, pn = pooled_contrast(F[t].values, df, adjust_steps=False)
                da, pa = pooled_contrast(F[t].values, df, adjust_steps=(dataset == "eedi"))
                pm_delta = {m: (Fm[m][t].values[permodel[m]["group"].values==1].mean()
                                - Fm[m][t].values[permodel[m]["group"].values==0].mean()) for m in MODELS}
                rows.append(dict(dataset=dataset, regime=regime, kind=kind, tag=t,
                                 mean_hi=F[t].values[df["group"].values==1].mean(),
                                 mean_lo=F[t].values[df["group"].values==0].mean(),
                                 delta=dn, p=pn, adj_delta=da, p_adj=pa,
                                 delta_ds=pm_delta["deepseek"], delta_glm=pm_delta["glm"]))
            sub = pd.DataFrame(rows)
            sub["q"] = bh_fdr(sub["p"].values); sub["q_adj"] = bh_fdr(sub["p_adj"].values)
            recs.append(sub)
    return pd.concat(recs, ignore_index=True) if recs else pd.DataFrame()

RES = run_stage2()

# report SciQ (or any un-gated setting) as null, not interpreted
for dataset in DATASETS:
    for regime in REGIMES:
        if (dataset, regime) not in GATED:
            pv = OMNI[(OMNI.dataset==dataset)&(OMNI.regime==regime)].p.values[0]
            print(f"[{dataset}/{regime}] omnibus p={pv:.3f} (ns) -> no overall difference detected; "
                  f"per-tag effects NOT interpreted.")
print("\nStage-2 rows:", len(RES))

In [ ]:
# ---------- decomposition tables (one per gated Eedi regime) ----------
def show_decomp(dataset, regime):
    sub = RES[(RES.dataset==dataset)&(RES.regime==regime)]
    if not len(sub): return
    def mk(kind):
        s = sub[sub.kind==kind].set_index("tag")
        return pd.DataFrame({
            f"{kind}:naive":   [f"{s.loc[t,'delta']:+.2f}{star(s.loc[t,'q'],s.loc[t,'p'])}" for t in TAGS],
            f"{kind}:stepadj": [f"{s.loc[t,'adj_delta']:+.2f}{star(s.loc[t,'q_adj'],s.loc[t,'p_adj'])}" for t in TAGS],
        }, index=[TAG_DISPLAY_NAMES[t] for t in TAGS])
    tbl = pd.concat([mk("count"), mk("share")], axis=1)
    # per-model sign agreement (share, naive)
    sh = sub[sub.kind=="share"].set_index("tag")
    tbl["share signs ds/glm"] = [f"{'+' if sh.loc[t,'delta_ds']>=0 else '-'}/"
                                 f"{'+' if sh.loc[t,'delta_glm']>=0 else '-'}" for t in TAGS]
    print(f"\n===== {dataset.upper()} / {REGIME_NAME[regime]} — Delta = good - bad "
          f"(count=abs; share=percentage points) =====")
    print(tbl.to_string())

def star(q, p):
    if np.isfinite(q) and q < 0.01: return "**"
    if np.isfinite(q) and q < 0.05: return "*"
    if np.isfinite(p) and p < 0.05: return "†"
    return ""

for (dataset, regime) in GATED:
    show_decomp(dataset, regime)
print("\n** BH q<0.01  * q<0.05  † nominal p<0.05 (uncorrected)")

In [ ]:
# ---------- heatmap of share Delta over gated settings ----------
def heatmap(kind, use_adj=False, fname=None):
    cols = GATED
    if not cols:
        print("no gated settings"); return
    M_ = np.full((len(TAGS), len(cols)), np.nan); ann = [["" for _ in cols] for _ in TAGS]
    dcol, qcol = ("adj_delta","q_adj") if use_adj else ("delta","q")
    for j,(d,r) in enumerate(cols):
        s = RES[(RES.dataset==d)&(RES.regime==r)&(RES.kind==kind)].set_index("tag")
        for i,t in enumerate(TAGS):
            if t in s.index:
                v = s.loc[t,dcol]; M_[i,j]=v
                if np.isfinite(v): ann[i][j]=f"{v:+.1f}"+star(s.loc[t,qcol],s.loc[t,'p'])
    vmax = np.nanmax(np.abs(M_)) if np.isfinite(M_).any() else 1.0
    fig, ax = plt.subplots(figsize=(1.6*len(cols)+2, 0.5*len(TAGS)+1.5))
    im = ax.imshow(M_, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels([f"{d}/{REGIME_NAME[r]}" for d,r in cols])
    ax.set_yticks(range(len(TAGS))); ax.set_yticklabels([TAG_DISPLAY_NAMES[t] for t in TAGS])
    for i in range(len(TAGS)):
        for j in range(len(cols)):
            if ann[i][j]:
                ax.text(j,i,ann[i][j],ha="center",va="center",fontsize=8,
                        color="black" if abs(M_[i,j])<0.6*vmax else "white")
    ax.set_title(f"{kind} contrast  Delta = good - bad (gated settings)"
                 + ("  [step-adj]" if use_adj else ""), fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03); fig.tight_layout()
    if fname: fig.savefig(f"figures/{fname}", bbox_inches="tight")
    plt.show()

heatmap("share", fname="behaviour_contrast_share.pdf")
heatmap("count", fname="behaviour_contrast_count.pdf")

## LaTeX for the appendix — omnibus table + gated per-tag tables

In [ ]:
# ---------- omnibus LaTeX table ----------
def omnibus_tex():
    lines = [r"\begin{table}[!htbp]", r"\centering", r"\small",
             r"\begin{tabular}{ll|r r}", r"\hline",
             r"\textbf{Dataset} & \textbf{Regime} & \textbf{pseudo-$F$} & \textbf{$p$} \\", r"\hline"]
    for _, row in OMNI.iterrows():
        pstr = f"{row.p:.3f}" + ("$^{*}$" if row.p < 0.05 else "")
        if np.isfinite(row.p_diffrobust):
            pstr += f" ({row.p_diffrobust:.3f})"
        lines.append(f"{row.dataset.capitalize()} & {REGIME_NAME[row.regime]} & {row.pseudoF:.2f} & {pstr} \\\\")
    lines += [r"\hline", r"\end{tabular}",
              r"\caption{Stage 1 omnibus: PERMANOVA (pseudo-$F$ on Bray--Curtis dissimilarity of the ten-strategy "
              r"count profile) for a difference between good (proportional match ${>}0.5$) and bad (${<}0.5$) traces, pooling both "
              r"models with restricted permutation within model. For Eedi the parenthesised $p$ is the "
              r"difficulty-robust variant (permutation restricted within model$\times$step-count tercile). "
              r"$^{*}p<0.05$.}", r"\label{tab:behaviour-omnibus}", r"\end{table}"]
    return "\n".join(lines)

def _sig_tex(q, p):
    if np.isfinite(q) and q < 0.01: return "^{**}"
    if np.isfinite(q) and q < 0.05: return "^{*}"
    if np.isfinite(p) and p < 0.05: return "^{\\dagger}"
    return ""

def cell_tex(delta, q, p):
    if not np.isfinite(delta): return r"\multicolumn{2}{c}{--}"
    a, b = f"{delta:+.2f}".split("."); return f"${a}$ & ${b}{_sig_tex(q,p)}$"

def posthoc_tex(dataset, regime):
    sub = RES[(RES.dataset==dataset)&(RES.regime==regime)]
    lines = [r"\begin{table}[!htbp]", r"\centering", r"\small",
             r"\begin{tabular}{l|r@{.}l r@{.}l|r@{.}l r@{.}l}", r"\hline",
             r"& \multicolumn{4}{c}{\textbf{count}} & \multicolumn{4}{c}{\textbf{share (pp)}} \\",
             r"\cmidrule(lr){2-5}\cmidrule(lr){6-9}",
             r"\textbf{Strategy} & \multicolumn{2}{c}{naive} & \multicolumn{2}{c}{step-adj.}"
             r" & \multicolumn{2}{c}{naive} & \multicolumn{2}{c}{step-adj.} \\", r"\hline"]
    for t in TAGS:
        c = sub[(sub.kind=="count")&(sub.tag==t)].iloc[0]
        s = sub[(sub.kind=="share")&(sub.tag==t)].iloc[0]
        cells = " & ".join([cell_tex(c.delta,c.q,c.p), cell_tex(c.adj_delta,c.q_adj,c.p_adj),
                            cell_tex(s.delta,s.q,s.p), cell_tex(s.adj_delta,s.q_adj,s.p_adj)])
        lines.append(f"{TAG_DISPLAY_NAMES[t]} & {cells} \\\\")
    lines += [r"\hline", r"\end{tabular}",
              r"\caption{Stage 2 (gated): good$-$bad $\Delta$ in per-trace strategy usage for %s / %s traces "
              r"(positive $=$ good traces use it more), pooling both models. \emph{count}: absolute occurrences; "
              r"\emph{share}: percentage of the trace's tags. \emph{step-adj.}: partials out "
              r"\texttt{nr\_steps\_of\_solution} (Freedman--Lane). Permutation test, Benjamini--Hochberg across "
              r"the ten strategies within each column ($^{*}q<0.05$, $^{**}q<0.01$; $^{\dagger}$ nominal "
              r"$p<0.05$).}" % (dataset.capitalize(), REGIME_NAME[regime]),
              r"\label{tab:behaviour-posthoc-%s-%s}" % (dataset, regime), r"\end{table}"]
    return "\n".join(lines)

print("% ===== OMNIBUS ====="); print(omnibus_tex()); print()
for (dataset, regime) in GATED:
    print(f"% ===== POST-HOC {dataset}/{regime} ====="); print(posthoc_tex(dataset, regime)); print()

## Stage-level view — strategy usage over the trace, split by performance

Performance-split version of the Figure 2 temporal panel -> Figure 9 / Figure 10.

In [ ]:
# ---------- stage-level strategy usage over the trace, split by high/low performance ----------
# Reuses the normalized-position binning of 051 (Figure 2). Each row is a prompting
# setting (reasoning / CoT) and each column a performance stratum (high / low), for one (dataset, model).
BINS = np.linspace(0, 1, 6)                       # five 20%-slices, as in the main-paper temporal panel
# Paul Tol muted palette in the taxonomy order used elsewhere in the paper
_TOL = {"inter":"#88CCEE","corr":"#117733","err_desc":"#CC6677","inst":"#DDCC77","err_sim":"#AA4499",
        "plaus":"#332288","curate":"#44AA99","recon":"#000000","link":"#999933","discr":"#882255"}
LEGEND_ORDER = ["inter","corr","link","err_desc","err_sim","inst","plaus","discr","curate","recon"]
MODEL_DISPLAY = {"deepseek":"DeepSeek", "glm":"GLM"}

def parse_positions(path):
    """Normalized (pos / len(trace)) positions for every tag occurrence in a run's annotated CSV."""
    assert "long_problems" not in path
    pos_by_tag = {t: [] for t in TAGS}
    df = pd.read_csv(path)
    for _, row in df.iterrows():
        trace = row.get("trace", ""); seq_str = row.get("annotation_sequence", "")
        seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
        L = len(trace) if isinstance(trace, str) and trace else 1
        for pos, tag in seq:
            if tag in pos_by_tag:
                pos_by_tag[tag].append(pos / L)
    return pos_by_tag, len(df)

def share_over_position(pos_by_tag, bins=BINS):
    """Per bin, share of each strategy among all tags falling in that bin (columns sum to 1)."""
    counts = {t: np.histogram(np.array(pos_by_tag[t]), bins=bins)[0] for t in TAGS}
    nb = len(bins) - 1; share = {t: np.zeros(nb) for t in TAGS}
    for b in range(nb):
        tot = sum(counts[t][b] for t in TAGS)
        if tot > 0:
            for t in TAGS:
                share[t][b] = counts[t][b] / tot
    return share, (bins[:-1] + bins[1:]) / 2

def _perf_path(dataset, model, regime, bucket):
    return f"{DATAFOLDER[dataset]}/joint_results/annotated/{STEMS[(model,regime)]}_{bucket}_match_solvable_annot_parsed.csv"

def _load_split(dataset, model, regime):
    hi_p, n_hi = parse_positions(_perf_path(dataset, model, regime, "high"))
    lo_p, n_lo = parse_positions(_perf_path(dataset, model, regime, "low"))
    hi_s, centers = share_over_position(hi_p); lo_s, _ = share_over_position(lo_p)
    return hi_s, lo_s, centers, n_hi, n_lo

def print_stage_diff(dataset, model, regime):
    hi_s, lo_s, centers, n_hi, n_lo = _load_split(dataset, model, regime)
    print(f"\n=== {model} / {REGIME_NAME[regime]} / {dataset} (high n={n_hi}, low n={n_lo}) — high-minus-low share per stage ===")
    print("    stage " + " ".join(f"{TAG_DISPLAY_NAMES[t][:9]:>9}" for t in TAGS))
    for b, c in enumerate(centers):
        print(f"    {c:5.1f} " + " ".join(f"{(hi_s[t][b]-lo_s[t][b]):+9.3f}" for t in TAGS))

def plot_perf_grid(dataset, model, save=True):
    """2x2 grid: rows = prompting setting (reasoning / CoT), cols = performance (high / low)."""
    fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey="row")
    for i, regime in enumerate(REGIMES):
        hi_s, lo_s, centers, n_hi, n_lo = _load_split(dataset, model, regime)
        for j, (share, stratum, n) in enumerate([(hi_s, "High", n_hi), (lo_s, "Low", n_lo)]):
            ax = axes[i, j]
            for t in TAGS:
                ax.plot(centers, share[t], marker="o", lw=2, ms=6, color=_TOL[t], label=TAG_DISPLAY_NAMES[t])
            ax.set_title(f"{REGIME_NAME[regime]} — {stratum} performance (n={n})", fontweight="bold")
            ax.grid(True, alpha=0.3); ax.set_xlim(BINS[0], BINS[-1])
            if j == 0: ax.set_ylabel("Share of Strategies", fontweight="bold")
            if i == 1: ax.set_xlabel("Normalized Position In Trace", fontweight="bold")
    fig.legend(handles=[plt.Line2D([0],[0], color=_TOL[t], marker="o", lw=2, ms=6, label=TAG_DISPLAY_NAMES[t])
                        for t in LEGEND_ORDER], loc="center left", bbox_to_anchor=(0.99, 0.5), frameon=True)
    fig.suptitle(f"Strategy usage in {MODEL_DISPLAY[model]} traces on {dataset.capitalize()}, split by performance",
                 fontweight="bold", y=1.00)
    fig.tight_layout()
    if save:
        out = f"figures/perf_split_{model}_{dataset}.pdf"
        fig.savefig(out, bbox_inches="tight", dpi=300); print("saved", out)
    plt.show()

for model in MODELS:
    for regime in REGIMES:
        print_stage_diff("eedi", model, regime)
    plot_perf_grid("eedi", model)